In [ ]:
import os
import numpy as np
from glob import glob
from pynwb import NWBHDF5IO
import scipy.io

In [ ]:
# Daume et al. 2024 (Nature) NWB dataset — Sternberg working memory task.
# NWB files are organised as one file per session under sub-* directories.
nwb_input_dir     = r'/path/to/NWB/dataset'
nwb_session_files = sorted(glob(os.path.join(nwb_input_dir, 'sub-*/*.nwb')))

# Output directory for preprocessed .mat files
lfp_process_dir = r'/path/to/output/preprocess'
os.makedirs(lfp_process_dir, exist_ok=True)

# NWB files include 10 s of padding applied during LFP generation; subtract
# this offset to recover timestamps relative to true trial onset.
LFP_PAD_us = 0  # seconds

# Additional padding added here around each trial window to reduce edge effects
edge_offset = 10.0  # seconds

In [ ]:
# Load LFP (micro) from each session NWB file and save as .mat.
# Each .mat contains: raw traces (channels x time), channel labels/coords/locations,
# encoding and recognition trial epoch boundaries, sampling rate, and timestamps.
# Note: 'OFC' = vmPFC and 'SMA' = preSMA.

for data2load in ['LFP_micro']:

    for session_ii in nwb_session_files:
        print(f'processing {data2load} {os.path.basename(session_ii)}...')

        with NWBHDF5IO(session_ii, 'r') as nwb_io:
            nwbfile = nwb_io.read()
            session_id = nwbfile.identifier

            trials_df = nwbfile.trials.to_dataframe()

            # Split trials by phase and add edge padding for epoch extraction
            enc_df   = trials_df[trials_df['stim_phase'] == 'encoding']
            recog_df = trials_df[trials_df['stim_phase'] == 'recognition']

            enc_start_time   = enc_df['start_time'].to_numpy()   - edge_offset
            enc_stop_time    = enc_df['stop_time'].to_numpy()    + edge_offset
            recog_start_time = recog_df['start_time'].to_numpy() - edge_offset
            recog_stop_time  = recog_df['stop_time'].to_numpy()  + edge_offset

            try:
                lfp_elect_series = nwbfile.processing['ecephys'][data2load]['ElectricalSeries']
            except KeyError:
                print(f"\n\tCouldn't find {data2load}. Skipped.\n")
                continue

            # Load and transpose to (channels x time)
            lfp_data = lfp_elect_series.data[:].T

            lfp_channel_info_df = lfp_elect_series.electrodes.to_dataframe()
            chan_labels    = lfp_channel_info_df['origchannel_name'].to_numpy(dtype=str)
            assert len(chan_labels) == lfp_data.shape[0]

            chan_coords = np.column_stack((
                lfp_channel_info_df['x'].to_numpy(dtype='float32'),
                lfp_channel_info_df['y'].to_numpy(dtype='float32'),
                lfp_channel_info_df['z'].to_numpy(dtype='float32'),
            ))
            chan_locations = lfp_channel_info_df['location'].to_numpy(dtype='str')

            # Reconstruct time vector; NWB files use either a rate or explicit timestamps
            if lfp_elect_series.rate is None:
                lfp_time = lfp_elect_series.timestamps[:]
                sfreq    = 1. / np.diff(lfp_time).mean()
            else:
                lfp_time = (np.arange(0, lfp_data.shape[1]) / lfp_elect_series.rate
                            + lfp_elect_series.starting_time)
                sfreq = lfp_elect_series.rate
            assert len(lfp_time) == lfp_data.shape[1]

            # Remove the pre-applied NWB padding to recover true timestamps
            lfp_time -= LFP_PAD_us

            mdic = {
                'lfp_data':         lfp_data,
                'chan_labels':      chan_labels,
                'chan_coords':      chan_coords,
                'chan_locations':   chan_locations,
                'enc_start_time':   enc_start_time,
                'enc_stop_time':    enc_stop_time,
                'recog_start_time': recog_start_time,
                'recog_stop_time':  recog_stop_time,
                'fs':               sfreq,
                'times':            lfp_time,
            }
            filename = os.path.join(lfp_process_dir, session_id + '_' + data2load + '.mat')
            scipy.io.savemat(filename, mdic)

In [ ]:
nwb_session_files = sorted(glob(os.path.join(nwb_input_dir, 'sub-*/*.nwb')))

# Load single-unit spike times and metadata from each session and save as .mat.
# electrode_id is incremented by 1 to convert from 0-indexed Python to 1-indexed MATLAB.
for session_ii in nwb_session_files:
    print(f'processing {os.path.basename(session_ii)}...')

    with NWBHDF5IO(session_ii, 'r') as nwb_io:
        nwbfile = nwb_io.read()
        session_id = nwbfile.identifier

        units_df = nwbfile.units.to_dataframe()

        # Convert to 1-based electrode indexing for MATLAB compatibility
        units_df['electrode_id'] = units_df['electrode_id'] + 1

        filename = os.path.join(lfp_process_dir, session_id + '_units.mat')
        scipy.io.savemat(filename, units_df)